In [115]:
import torch
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

In [116]:
labels = torch.load('./beats/labels.pt')
data = pl.read_csv('ds/b05_onlykw_onlyexists.csv')

In [117]:
def get_frequencies(unique_labels, sample):
    counts = np.zeros_like(unique_labels)
    for i, v in enumerate(unique_labels):
        for w in sample:
            if v==w:
                counts[i] += 1
    return counts

In [118]:
labels = {k: v.numpy() for k, v in labels.items()}
unique_labels = set()
for v in labels.values():
    for w in v:
        unique_labels.add(int(w))
unique_labels=sorted(list(unique_labels))

In [119]:
new_labels = defaultdict(dict)
for k, v in labels.items():
    new_labels[k]['labels'] = v
    new_labels[k]['freq'] = get_frequencies(unique_labels, v)
    new_labels[k]['ecotype'] = data['Ecotype'][k]


In [123]:
X = [new_labels[k]['freq'] for k in new_labels.keys()]
y = [new_labels[k]['ecotype'] for k in new_labels.keys()]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [124]:
clf = LogisticRegression(random_state=59, max_iter=300).fit(X_freq, y)

In [125]:
print(clf.score(X_train, y_train))
print(clf.score(X_test, y_test))

0.9846272098385856
0.9844590555887627


In [141]:
data_per_dataset=data.pivot(on="Ecotype", index="Dataset", values="Ecotype", aggregate_function="len")

In [151]:
result = (
    data_per_dataset.with_columns(
        (pl.col('SRKW') - pl.col('TKW')).abs().alias("diff")
    )
    .filter((pl.col("diff") > 0 ) ^ (pl.col("SRKW") == 0) ^ (pl.col("TKW") == 0))
    .sort("diff")
    .head(1)
)
print(result["Dataset"])

shape: (1,)
Series: 'Dataset' [str]
[
	"CarmanahPt"
]


In [152]:
data_per_dataset.filter(pl.col("Dataset")=="CarmanahPt")

Dataset,SRKW,TKW
str,u32,u32
"""CarmanahPt""",132,55


In [153]:
X = [new_labels[k]['freq'] for k in new_labels.keys() if data["Dataset"][k] != "CarmanahPt"]
y = [new_labels[k]['ecotype'] for k in new_labels.keys() if data["Dataset"][k]!= "CarmanahPt"]
X_test = [new_labels[k]['freq'] for k in new_labels.keys() if data["Dataset"][k] == "CarmanahPt"]
y_test = [new_labels[k]['ecotype'] for k in new_labels.keys() if data["Dataset"][k]== "CarmanahPt"]

In [155]:
clf = LogisticRegression(random_state=59, max_iter=300).fit(X, y)
print(clf.score(X, y))
print(clf.score(X_test, y_test))

0.9896084616812024
0.6951871657754011
